# 14 — RAG with Reranking

Two-stage retrieval: broad vector search → LLM-based reranking → answer generation.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

## Documents and Reranker

In [ ]:
DOCUMENTS = [
    Document(page_content="Python 3.12 introduced several performance improvements, including a new specializing adaptive interpreter.", metadata={"source": "python_release"}),
    Document(page_content="Django 5.0 added facet filters in the admin, simplified form field rendering, and database-generated model fields.", metadata={"source": "django_release"}),
    Document(page_content="FastAPI is a modern web framework for building APIs with Python, based on standard Python type hints.", metadata={"source": "fastapi_overview"}),
    Document(page_content="Flask is a lightweight WSGI web framework. It is designed with simplicity and extensibility in mind.", metadata={"source": "flask_overview"}),
    Document(page_content="Python's GIL prevents true multi-threading for CPU-bound tasks. Python 3.13 experiments with a no-GIL build.", metadata={"source": "python_gil"}),
    Document(page_content="NumPy 2.0 introduced a new string dtype, improved type promotion rules, and removed many deprecated features.", metadata={"source": "numpy_release"}),
    Document(page_content="LangChain's expression language (LCEL) allows composing chains with the pipe operator for declarative workflows.", metadata={"source": "langchain_lcel"}),
    Document(page_content="Pydantic V2 is a ground-up rewrite in Rust, offering 5-50x speed improvements over V1 for data validation.", metadata={"source": "pydantic_v2"}),
]

class RerankedDoc(BaseModel):
    index: int = Field(description="Original index of the document")
    relevance: float = Field(description="Relevance score 0.0 to 1.0")

class RerankedResults(BaseModel):
    documents: list[RerankedDoc]

def rerank_documents(query, docs, llm, top_k=3):
    doc_descriptions = "\n".join(f"[{i}] {doc.page_content[:200]}" for i, doc in enumerate(docs))
    prompt = ChatPromptTemplate.from_template(
        "Given the query, score each document's relevance from 0.0 to 1.0.\n\nQuery: {query}\n\nDocuments:\n{documents}"
    )
    result = (prompt | llm.with_structured_output(RerankedResults)).invoke({"query": query, "documents": doc_descriptions})
    ranked = sorted(result.documents, key=lambda d: d.relevance, reverse=True)
    return [(docs[r.index], r.relevance) for r in ranked[:top_k] if r.index < len(docs)]

## Retrieve, Rerank, Answer

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
vectorstore = Chroma.from_documents(DOCUMENTS, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

for query in ["What performance improvements were made to Python recently?", "Which web frameworks are available for Python?"]:
    print(f"Q: {query}")
    candidates = retriever.invoke(query)
    reranked = rerank_documents(query, candidates, llm, top_k=2)
    for doc, score in reranked:
        print(f"  [{score:.2f}] {doc.page_content[:80]}...")
    context = "\n\n".join(doc.page_content for doc, _ in reranked)
    answer = (ChatPromptTemplate.from_template("Answer using context:\n{context}\n\nQuestion: {question}") | llm | StrOutputParser()).invoke({"context": context, "question": query})
    print(f"  Answer: {answer}\n")

vectorstore.delete_collection()